**Setting up Spark Environment**

1\. Deploy a spark Cluster (Dataproc)

2\. Store Data in HDFS (Hadoop Distributed File System) and not in local.

*   Download the data from Souce Kaggle:[https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
*   curl command : #!/bin/bash curl -L -o ~/olist/brazilian-ecommerce.zip\
  https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce
*   unzip brazilian-ecommerce.zip -d ~/olist/data/

3\. Use Pyspark to interact with Data.


In [25]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('OlistData').getOrCreate()

In [5]:
spark

In [6]:
!hadoop fs -ls /data/olist/

Found 9 items
-rw-r--r--   2 leynardlove hadoop    9033957 2026-03-06 09:18 /data/olist/olist_customers_dataset.csv
-rw-r--r--   2 leynardlove hadoop   61273883 2026-03-06 09:19 /data/olist/olist_geolocation_dataset.csv
-rw-r--r--   2 leynardlove hadoop   15438671 2026-03-06 09:19 /data/olist/olist_order_items_dataset.csv
-rw-r--r--   2 leynardlove hadoop    5777138 2026-03-06 09:19 /data/olist/olist_order_payments_dataset.csv
-rw-r--r--   2 leynardlove hadoop   14451670 2026-03-06 09:19 /data/olist/olist_order_reviews_dataset.csv
-rw-r--r--   2 leynardlove hadoop   17654914 2026-03-06 09:19 /data/olist/olist_orders_dataset.csv
-rw-r--r--   2 leynardlove hadoop    2379446 2026-03-06 09:19 /data/olist/olist_products_dataset.csv
-rw-r--r--   2 leynardlove hadoop     174703 2026-03-06 09:19 /data/olist/olist_sellers_dataset.csv
-rw-r--r--   2 leynardlove hadoop       2613 2026-03-06 09:19 /data/olist/product_category_name_translation.csv


In [10]:
hdfs_path = '/data/olist/'

customer_df = spark.read.csv(hdfs_path + 'olist_customers_dataset.csv', header=True, inferSchema=True)
order_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv', header=True, inferSchema=True)
order_items_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv', header=True, inferSchema=True)
payment_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv', header=True, inferSchema=True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv', header=True, inferSchema=True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv', header=True, inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv', header=True, inferSchema=True)
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv', header=True, inferSchema=True)

In [20]:
# Check Data Lekage or Drop

print(f'Customers : {customer_df.count()} rows')
print(f'Orders : {order_df.count()} rows')
print(f'Order Items : {order_items_df.count()} rows')
print(f'Payments : {payment_df.count()} rows')
print(f'Products : {products_df.count()} rows')
print(f'GeoLocation : {geolocation_df.count()} rows')
print(f'Category Translation : {category_translation_df.count()} rows')

Customers : 99441 rows
Orders : 99441 rows
Order Items : 112650 rows
Payments : 103886 rows
Products : 32951 rows
GeoLocation : 1000163 rows
Category Translation : 71 rows


In [81]:
## Check Null in every columns

customer_null_counts = [F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in customer_df.columns]
orders_null_counts = [F.count(F.when(F.col(c).isNull(), 1)).alias(c)  for c in order_df.columns]
order_items_null_counts = [F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in order_items_df.columns]
payment_null_counts = [F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in payment_df.columns]
products_null_counts = [F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in products_df.columns]
geolocation_null_counts = [F.count(F.when(F.col(c).isNull(),1)).alias(c) for c in geolocation_df.columns]
category_translation_null_counts = [F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in category_translation_df.columns]

print('================Customer DataFrame Count column nulls===============')
customer_df.select(customer_null_counts).show()

print('================Order DataFrame Count column nulls===============')
order_df.select(orders_null_counts).show()

print('================Order Items DataFrame Count column nulls===============')
order_items_df.select(order_items_null_counts).show()

print('================Payment DataFrame Count column nulls===============')
payment_df.select(payment_null_counts).show()

print('================Products DataFrame Count column nulls===============')
products_df.select(products_null_counts).show()

print('================Geolocation DataFrame Count column nulls===============')
geolocation_df.select(geolocation_null_counts).show()

print('================Category Translation DataFrame Count column nulls===============')
category_translation_df.select(category_translation_null_counts).show()

================Customer DataFrame Count column nulls===============
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+

================Order DataFrame Count column nulls===============
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+---------------

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+

================Category Translation DataFrame Count column nulls===============
+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



In [82]:
# Check Duplicates Values

print('=== Check Duplicates in customer dataframe columns ===')
customer_df.groupby('customer_id').count().filter(F.col('count')>1).show()

print('=== Check Duplicates in order dataframe columns ===')
order_df.groupBy('order_id').count().filter(F.col('count')>1).show()

print('=== Check Duplicates in order items dataframe columns ===')
order_items_df.groupBy('order_id').count().filter(F.col('count')>1).show()

print('=== Check Duplicates in payment dataframe columns ===')
payment_df.groupBy('order_id').count().filter(F.col('count')>1).show()

print('=== Check Duplicates in products dataframe columns ===')
products_df.groupBy('product_id').count().filter(F.col('count')>1).show()

print('=== Check Duplicates in geolocation dataframe columns ===')
geolocation_df.groupBy('geolocation_zip_code_prefix').count().filter(F.col('count')>1).show()

print('=== Check Duplicates in category translation dataframe columns ===')
category_translation_df.groupBy('product_category_name').count().filter(F.col('count')>1).show()

=== Check Duplicates in customer dataframe columns ===
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+

=== Check Duplicates in order dataframe columns ===
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+

=== Check Duplicates in order items dataframe columns ===
+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|0baa56401ae212e30...|    2|
|118045506e1c1dda0...|    5|
|1c4a92d82c1b0dec1...|    3|
|414a0b1cbb58b3ae7...|    2|
|502d7a6ef832644f5...|    2|
|7a5472f7c8cecc2e1...|    2|
|1728847cf9a9a8c5f...|    2|
|1e48885ebbfefb5b4...|    2|
|95087a791500e383c...|    2|
|055cf2a6f9343f99a...|    6|
|0cbea266e238cccbc...|    2|
|2fc4ba497db1306b9...|    2|
|3806e97292af7b943...|    2|
|533e799c38f442b59...|    2|
|986da3ca575a68c69...|    2|
|a24474d0a4ebace56...|    2|
|0b127810f86631fa8...|    2|
|15ccd946546561439...|    2|
|20db59f6705840a0c...|    2|
|5bd04e944fbdeb94c...|    2|
+-----------------

+---------------------------+-----+
|geolocation_zip_code_prefix|count|
+---------------------------+-----+
|                       2122|   33|
|                       2366|   33|
|                       3918|   50|
|                       4101|   72|
|                       9852|  107|
|                      13289|   61|
|                      26087|  111|
|                      28024|   95|
|                       3226|   24|
|                       4190|   52|
|                      18201|   69|
|                      20396|   11|
|                       1303|  166|
|                       6825|   22|
|                       8257|   60|
|                      13483|   83|
|                      24855|   41|
|                      25638|    2|
|                       2443|   80|
|                       2721|  149|
+---------------------------+-----+
only showing top 20 rows

=== Check Duplicates in category translation dataframe columns ===
+---------------------+-----+
|product_cate

In [48]:
# Customer Distribution by State

customer_df.groupBy('customer_state').count().orderBy(F.col('count'), ascending=False).show()

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
|            PE| 1652|
|            CE| 1336|
|            PA|  975|
|            MT|  907|
|            MA|  747|
|            MS|  715|
|            PB|  536|
|            PI|  495|
|            RN|  485|
|            AL|  413|
+--------------+-----+
only showing top 20 rows



In [51]:
# Order - Order status distribution

order_df.groupBy('order_status').count().orderBy(F.col('count'), ascending=False).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [53]:
# Payments

payment_df.groupBy('payment_type').count().orderBy(F.col('count'), ascending=False).show()

+------------+-----+
|payment_type|count|
+------------+-----+
| credit_card|76795|
|      boleto|19784|
|     voucher| 5775|
|  debit_card| 1529|
| not_defined|    3|
+------------+-----+



In [68]:
# Top selling Products

top_product = order_items_df.groupBy('product_id').agg(F.sum(F.col('price')).alias('total_sales'))
top_product.orderBy('total_sales', ascending=False).show()

+--------------------+------------------+
|          product_id|       total_sales|
+--------------------+------------------+
|bb50f2e236e5eea01...|           63885.0|
|6cdd53843498f9289...| 54730.20000000005|
|d6160fb7873f18409...|48899.340000000004|
|d1c427060a0f73f6b...| 47214.51000000006|
|99a4788cb24856965...|43025.560000000085|
|3dd2a17168ec895c7...| 41082.60000000005|
|25c38557cf793876c...| 38907.32000000001|
|5f504b3a1c75b73d6...|37733.899999999994|
|53b36df67ebb7c415...| 37683.42000000001|
|aca2eb7d00ea1a7b8...| 37608.90000000007|
|e0d64dcfaa3b6db5c...|          31786.82|
|d285360f29ac7fd97...|31623.809999999983|
|7a10781637204d8d1...|           30467.5|
|f1c7f353075ce59d8...|          29997.36|
|f819f0c84a64f02d3...|29024.479999999996|
|588531f8ec37e7d5f...|28291.989999999998|
|422879e10f4668299...|26577.219999999972|
|16c4e87b98a9370a9...|           25034.0|
|5a848e4ab52fd5445...|24229.029999999962|
|a62e25e09e05e6faf...|           24051.0|
+--------------------+------------

In [80]:
# Total Delivery Time Analysis

delivery_df = order_df.select('order_id', 'order_purchase_timestamp', 'order_delivered_customer_date')

delivery_detail_df = delivery_df.withColumn('delivery_time', F.datediff(F.col('order_delivered_customer_date'),  F.col('order_purchase_timestamp')))

delivery_detail_df.orderBy('delivery_time', ascending=False).show()


+--------------------+------------------------+-----------------------------+-------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|delivery_time|
+--------------------+------------------------+-----------------------------+-------------+
|ca07593549f1816d2...|     2017-02-21 23:31:27|          2017-09-19 14:36:39|          210|
|1b3190b2dfa9d789e...|     2018-02-23 14:57:35|          2018-09-19 23:24:07|          208|
|440d0d17af552815d...|     2017-03-07 23:59:51|          2017-09-19 15:12:50|          196|
|2fb597c2f772eca01...|     2017-03-08 18:09:02|          2017-09-19 14:33:17|          195|
|285ab9426d6982034...|     2017-03-08 22:47:40|          2017-09-19 14:00:04|          195|
|0f4519c5f1c541dde...|     2017-03-09 13:26:57|          2017-09-19 14:38:21|          194|
|47b40429ed8cce3ae...|     2018-01-03 09:44:01|          2018-07-13 20:51:31|          191|
|2fe324febf907e3ea...|     2017-03-13 20:17:10|          2017-09-19 17:00:07|   